# ML HOL - PART 1

## ML Data Ingestion

- In this notebook, we will walk through how to establish a connection to a Snowflake.
- We will perform a basic ingestion task that writes out a CSV file into a raw table.

***Note: This notebook will use the `diamonds` dataset, a widely used in data science and machine learning. We will use it to demonstrate Snowflake's native data science transformers in terms of database functionality and Spark & Pandas comportablity, using non-synthetic and statistically appropriate data that is well known to the ML community.***



## Pre-installed Packages

This notebook runs on **Container Runtime**, which is powered by Snowpark Container Services. The runtime image comes pre-installed with common data science packages including `snowflake-snowpark-python`, so no additional package installation is needed for this notebook.

When creating this notebook, ensure you selected **Run on container** for the Runtime, chose a **CPU** compute pool, and selected an appropriate warehouse for SQL execution.

Once your container session is **Active**, you're ready to run the rest of the Notebook.

***Note: SQL and Snowpark queries are still pushed down to the warehouse for optimized performance. The compute pool runs the Python kernel.***

***Container Runtime details: https://docs.snowflake.com/en/developer-guide/snowflake-ml/notebooks-on-spcs***

### Import Libraries

In [ ]:
# Snowpark for Python
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import DoubleType
import snowflake.snowpark.functions as F

### Setup and establish Secure Connection to Snowflake

Notebooks on Container Runtime automatically establish a Snowpark Session when the notebook connects to its compute pool. SQL and Snowpark queries are pushed down to the assigned warehouse. We will use an assigned warehouse, database, and schema throughout this tutorial.

In [ ]:
# Get Snowflake Session object
session = get_active_session()
session.sql_simplifier_enabled = True

# Set the database and schema context
session.use_database('ML_HOL_DB')
user_suffix = ''.join(filter(str.isdigit, session.get_current_user()))
session.use_schema(f'ML_SCHEMA{user_suffix}')

# Add a query tag to the session.
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"e2e_ml_snowparkpython", 
                     "version":{"major":1, "minor":0,},
                     "attributes":{"is_hol":1, "source":"notebook"}}

# Current Environment Details
print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

### 1. Use the Snowpark DataFrame Reader to read in data from a `diamonds` CSV file 

Prior to HOL we staged the `diamonds.csv` file from an external s3 bucket and exposed it for you in an external stage (@ML_HOL_DB.PUBLIC.DIAMONDS_ASSETS).  Now, we can read it in.

For more information on loading data, see documentation on [snowflake.snowpark.DataFrameReader](https://docs.snowflake.com/ko/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrameReader.html).




In [ ]:
# Create a Snowpark DataFrame that is configured to load data from the CSV file
# We can now infer schema from CSV files.
diamonds_df = session.read.options({"field_delimiter": ",",
                                    "field_optionally_enclosed_by": '"',
                                    "infer_schema": True,
                                    "parse_header": True}).csv("@ML_HOL_DB.PUBLIC.DIAMONDS_ASSETS")

# Add the new column to the DataFrame
diamonds_df = diamonds_df.withColumn("DIAMONDS_ID", F.monotonically_increasing_id())
diamonds_df

### 2. Write out the raw data to a Snowflake table

There are many tools/options for loading data.  This is just one.

***Note: Loading Data summary: https://docs.snowflake.com/en/guides-overview-loading-data***

In [ ]:
diamonds_df.write.mode('overwrite').save_as_table('raw_diamonds')

***In the next notebook, we will perform data transformations and schedule them in a Feature Store.***